# Scientific Reports successor — Step 9

Constitutive and numerical robustness with a fail-closed claim lock. The Step 8 law is consumed unchanged.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import shutil, subprocess
from pathlib import Path

BRANCH = 'successor/scirep-waveform-susceptibility'
REPOSITORY = 'https://github.com/khalid-saqr/picoNewton.git'
CHECKOUT = Path('/content/picoNewton')
if CHECKOUT.exists():
    shutil.rmtree(CHECKOUT)
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPOSITORY, str(CHECKOUT)], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(CHECKOUT / 'picoNewton_v3')], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', str(CHECKOUT / 'piconewton_susceptibility')], check=True)


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/piconewton_susceptibility_outputs')
STEP8_ROOT = DRIVE_ROOT / 'step8_reduced_law'
STEP9_ROOT = DRIVE_ROOT / 'step9_robustness_claim_lock'
STEP9_ROOT.mkdir(parents=True, exist_ok=True)
required = ['step8_gate.json', 'step8_manifest.json', 'reduced_law.json', 'step8_reduced_law.npz']
missing = [name for name in required if not (STEP8_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing Step 8 artefacts: {missing}')


In [ ]:
import subprocess

subprocess.run([
    'piconewton-susceptibility-step9',
    '--step8-root', str(STEP8_ROOT),
    '--output', str(STEP9_ROOT),
    '--profile', 'publication',
], check=True)


In [ ]:
import json

manifest_path = STEP9_ROOT / 'step9_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
assert manifest['status'] == 'complete'
assert manifest['allowed_next_step'] == 10
assert manifest['gates']['passed'] is True
manifest


In [ ]:
claim_lock = json.loads((STEP9_ROOT / 'claim_lock.json').read_text(encoding='utf-8'))
assert claim_lock['status'] == 'locked'
assert claim_lock['allowed_next_step'] == 10
claim_lock


In [ ]:
import hashlib

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

verified = {}
for name, record in manifest['files'].items():
    path = STEP9_ROOT / name
    verified[name] = path.is_file() and sha256(path) == record['sha256']
assert all(verified.values())
verified


## Completion boundary

A passing notebook authorises Step 10 only. It does not broaden the reciprocal amplitude claim or create a biological interpretation.
